# Task 2: Ground-Up Vector Retrieval Engine

Goal: given a text query, find the most relevant chunks using **cosine similarity written from the formula**,
with no search library involved — only `numpy.dot` / `torch.matmul`.

$$\text{Similarity}(q, d) = \frac{q \cdot d}{|q|\,|d|}$$

Inputs come from Task 1 (`chunking_embeddings.ipynb`): 344 chunks and their `(344, 384)` embedding matrix.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import time

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from raghood.scrape import ROOT

PROCESSED_DIR = ROOT / "data" / "processed"

D:\Coding Stuff\rag-under-hood\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the corpus and its embeddings

`chunks.jsonl` is ordered by `id`, and `id` is the row number in `embeddings.npy`. Assert it rather than trust it:
a silent off-by-one here would make every result below subtly wrong and still look plausible.

In [2]:
chunks = [json.loads(line) for line in open(PROCESSED_DIR / "chunks.jsonl", encoding="utf-8")]
D = np.load(PROCESSED_DIR / "embeddings.npy")

assert [c["id"] for c in chunks] == list(range(len(chunks))), "chunks.jsonl is not ordered by id"
assert D.shape[0] == len(chunks), f"{D.shape[0]} vectors but {len(chunks)} chunks"

cdf = pd.DataFrame(chunks)
print(f"chunks     : {len(chunks)} from {cdf.page.nunique()} pages")
print(f"embeddings : {D.shape} {D.dtype}")

chunks     : 344 from 71 pages
embeddings : (344, 384) float32


In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                            device="cuda" if torch.cuda.is_available() else "cpu")
print(f"query encoder on {model.device}, {model.get_embedding_dimension()}-d output")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5682.89it/s]

query encoder on cuda:0, 384-d output


## 2. Cosine similarity, written out

The literal formula, with nothing hidden:

- `D @ q` is the dot product of the query against every chunk at once — one matrix-vector product,
  `(N, 384) @ (384,) -> (N,)`. This is the `q · d` numerator for all N documents.
- `np.linalg.norm(D, axis=1)` is `|d|` per row; `np.linalg.norm(q)` is `|q|`.
- Dividing gives the cosine of the angle between the query and each chunk, in `[-1, 1]`.

Cosine, not raw dot product, is what we want: it measures **direction only**, so a long chunk does not
score higher just because its vector happens to be longer.

In [ ]:
def cosine_similarity_numpy(q: np.ndarray, D: np.ndarray) -> np.ndarray:
    """Cosine similarity between query vector q (384,) and every row of D (N, 384).

    Straight from the definition: (q . d) / (|q| |d|).
    """
    dots = D @ q                                  # q . d for every d in D
    norms = np.linalg.norm(D, axis=1) * np.linalg.norm(q)   #  |d| * |q|
    return dots / norms

In [5]:
def cosine_similarity_torch(q: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    """Same formula with torch.matmul, so it can run on the GPU."""
    dots = torch.matmul(D, q)
    norms = torch.linalg.norm(D, dim=1) * torch.linalg.norm(q)
    return dots / norms

### Verify against scikit-learn

Our function must agree with a trusted implementation. `sklearn.metrics.pairwise.cosine_similarity` is the
reference; we only use it here as a test oracle, never in the retrieval path.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

rng = np.random.default_rng(0)
q_demo = model.encode("How do I register a blueprint?", convert_to_numpy=True)

mine = cosine_similarity_numpy(q_demo, D)
theirs = sk_cosine(q_demo.reshape(1, -1), D).ravel()
print(f"numpy vs sklearn : max abs diff {np.abs(mine - theirs).max():.2e}")
assert np.allclose(mine, theirs, atol=1e-6)

mine_t = cosine_similarity_torch(
    torch.from_numpy(q_demo).to(model.device), torch.from_numpy(D).to(model.device)
).cpu().numpy()
print(f"torch vs sklearn : max abs diff {np.abs(mine_t - theirs).max():.2e}")
assert np.allclose(mine_t, theirs, atol=1e-5)   # float32 GPU maths, slightly looser tolerance

# also check on deliberately un-normalised random vectors, where the denominator actually matters
R = rng.normal(size=(50, 384)).astype(np.float32) * rng.uniform(0.1, 10, (50, 1)).astype(np.float32)
qr = rng.normal(size=384).astype(np.float32) * 7
assert np.allclose(cosine_similarity_numpy(qr, R), sk_cosine(qr.reshape(1, -1), R).ravel(), atol=1e-5)
print("\nboth implementations match sklearn, including on un-normalised vectors")

numpy vs sklearn : max abs diff 8.94e-08
torch vs sklearn : max abs diff 5.96e-08

both implementations match sklearn, including on un-normalised vectors


### Why the denominator is a no-op on *our* data

Task 1 showed that `all-MiniLM-L6-v2` ends with a `Normalize` module, so every chunk vector already has
`|d| = 1`, and so does the query vector. The denominator is therefore `1 x 1 = 1`, and

$$\text{cosine}(q, d) = \frac{q \cdot d}{1 \cdot 1} = q \cdot d$$

i.e. the entire search is **one matrix-vector product**. Worth confirming numerically rather than assuming,
because it is the reason FAISS's `IndexFlatIP` (inner product) is the correct index to benchmark against in Task 5.

In [7]:
print(f"chunk norms |d| : min {np.linalg.norm(D, axis=1).min():.6f}  max {np.linalg.norm(D, axis=1).max():.6f}")
print(f"query norm  |q| : {np.linalg.norm(q_demo):.6f}")

dot_only = D @ q_demo
print(f"\nmax |cosine - plain dot product| : {np.abs(mine - dot_only).max():.2e}")
assert np.allclose(mine, dot_only, atol=1e-6)
print("on this corpus, cosine == dot product (keep the division anyway: it costs ~nothing")
print("and keeps the function correct if we ever feed it un-normalised vectors)")

chunk norms |d| : min 1.000000  max 1.000000
query norm  |q| : 1.000000

max |cosine - plain dot product| : 5.96e-08
on this corpus, cosine == dot product (keep the division anyway: it costs ~nothing
and keeps the function correct if we ever feed it un-normalised vectors)


## 3. Top-k selection

Sorting all N scores is wasteful when only k are needed. `np.argpartition` does a partial selection in
O(N) instead of O(N log N), then we sort just those k. At N=344 the difference is noise; at the N=5000
corpus of Task 5 it is the honest way to implement exhaustive search, and it makes the O(N) vs O(log N)
contrast with ANN indexes (Task 3) a fair one.

In [8]:
def top_k(scores: np.ndarray, k: int = 3) -> np.ndarray:
    """Indices of the k highest scores, best first."""
    k = min(k, scores.shape[0])
    idx = np.argpartition(-scores, k - 1)[:k]     # k best, unordered  -> O(N)
    return idx[np.argsort(-scores[idx])]          # order just those k -> O(k log k)


def search(query: str, k: int = 3) -> pd.DataFrame:
    """Embed the query, score it against every chunk, return the k best with their metadata."""
    q = model.encode(query, convert_to_numpy=True)
    scores = cosine_similarity_numpy(q, D)
    idx = top_k(scores, k)
    return pd.DataFrame([
        {"rank": r + 1, "score": float(scores[i]), "id": int(i),
         "page": chunks[i]["page"], "section": chunks[i]["section"],
         "url": chunks[i]["url"], "text": chunks[i]["text"]}
        for r, i in enumerate(idx)
    ])

In [9]:
# argpartition must agree with a full sort
s = cosine_similarity_numpy(q_demo, D)
assert list(top_k(s, 10)) == list(np.argsort(-s)[:10])
print("top_k matches a full argsort")
print(f"top-3 scores: {s[top_k(s, 3)].round(4)}")

top_k matches a full argsort


top-3 scores: [0.4864 0.406  0.4022]


## 4. Top-3 retrieval for the test queries

Three questions whose answers live on known pages, so the results can be judged by eye:

In [10]:
QUERIES = [
    "How do I register a blueprint?",
    "What is the application context?",
    "How to handle file uploads?",
]


def show(query: str, k: int = 3, chars: int = 420):
    res = search(query, k)
    print("=" * 78)
    print(f"QUERY: {query}")
    print("=" * 78)
    for r in res.itertuples():
        print(f"\n[{r.rank}] score {r.score:.4f}   {r.page}  >  {r.section}   (chunk {r.id})")
        print(f"    {r.url}")
        body = " ".join(r.text.split())
        print("    " + body[:chars] + (" ..." if len(body) > chars else ""))
    print()


for q in QUERIES:
    show(q)

QUERY: How do I register a blueprint?

[1] score 0.4864   tutorial/views  >  Create a Blueprint   (chunk 320)
    https://flask.palletsprojects.com/en/3.1.x/tutorial/views/
    from flaskr.db import get_db bp = Blueprint('auth', __name__, url_prefix='/auth') ``` This creates a Blueprint named `'auth'`. Like the application object, the blueprint needs to know where it's defined, so `__name__` is passed as the second argument. The `url_prefix` will be prepended to all the URLs associated with the blueprint. Import and register the blueprint from the factory using app.register_blueprint(). Pla ...

[2] score 0.4060   blueprints  >  Registering Blueprints   (chunk 24)
    https://flask.palletsprojects.com/en/3.1.x/blueprints/
    one is obviously from the application itself for the static files. The other two are for the `show` function of the `simple_page` blueprint. As you can see, they are also prefixed with the name of the blueprint and separated by a dot (`.`). Blueprints however can 

### Summary table

In [11]:
rows = []
for q in QUERIES:
    for r in search(q, 3).itertuples():
        rows.append({"query": q if r.rank == 1 else "", "rank": r.rank,
                     "score": round(r.score, 4), "page": r.page, "section": r.section})
pd.DataFrame(rows)

,query,rank,score,page,section
0,How do I register a blueprint?,1,0.4864,tutorial/views,Create a Blueprint
1,,2,0.4060,blueprints,Registering Blueprints
2,,3,0.4022,tutorial/views,Endpoints and URLs
3,What is the application context?,1,0.4356,appcontext,The Application Context
4,,2,0.4310,appcontext,Purpose of the Context
5,,3,0.3967,blueprints,Modular Applications with Blueprints
6,How to handle file uploads?,1,0.5671,patterns/fileuploads,Uploading Files
7,,2,0.5650,patterns/fileuploads,A Gentle Introduction
8,,3,0.5055,patterns/fileuploads,An Easier Solution


### Reading the results honestly

All three queries return chunks from the right pages, and the file-upload query is a clean sweep of
`patterns/fileuploads`. Two observations that are worth carrying into Task 4 rather than glossing over:

1. **"How do I register a blueprint?" ranks the tutorial above the reference.** Rank 1 is
   `tutorial/views > Create a Blueprint` (0.486); the dedicated `blueprints > Registering Blueprints`
   section is only rank 2 (0.406). Both are defensible, but a user asking that question probably wants the
   reference page. This is exactly the failure mode cross-encoder re-ranking is supposed to fix.
2. **Even the best hits only score ~0.4–0.57.** That is not a bug: `all-MiniLM-L6-v2` was trained on
   *symmetric* sentence-similarity pairs, but retrieval is *asymmetric* — a 7-word question against a
   200-word passage. The absolute numbers are therefore compressed, and only their *ordering* is
   meaningful. It also means a fixed score threshold ("only answer above 0.7") would reject every correct
   answer in this corpus.

## 5. Does it behave sensibly?

Two checks beyond "the output looks reasonable":

1. **Self-retrieval** — searching with a chunk's own text must return that chunk first, with a score of ~1.0.
   If this fails, the embedding matrix and the chunk list are misaligned.
2. **Score separation** — a real question should score its best hit clearly above an unrelated question's.

In [12]:
hits = 0
for i in [0, 100, 217, 343]:
    res = search(chunks[i]["text"], k=1)
    ok = res.id[0] == i
    hits += ok
    print(f"chunk {i:>3} -> retrieved {res.id[0]:>3}  score {res.score[0]:.4f}  {'ok' if ok else 'MISMATCH'}")
assert hits == 4, "self-retrieval failed: embeddings and chunks are misaligned"

chunk   0 -> retrieved   0  score 1.0000  ok
chunk 100 -> retrieved 100  score 1.0000  ok
chunk 217 -> retrieved 217  score 1.0000  ok
chunk 343 -> retrieved 343  score 1.0000  ok


In [13]:
probe = {
    "on-topic  : How do I register a blueprint?": "How do I register a blueprint?",
    "on-topic  : how to upload a file":          "how to upload a file",
    "off-topic : how to bake sourdough bread":   "how to bake sourdough bread",
    "off-topic : stock market prices today":     "stock market prices today",
}
for label, q in probe.items():
    res = search(q, 3)
    print(f"{label:<45} best {res.score[0]:.3f}  ({res.page[0]})")

on-topic  : How do I register a blueprint?    best 0.486  (tutorial/views)
on-topic  : how to upload a file              best 0.494  (patterns/fileuploads)
off-topic : how to bake sourdough bread       best 0.162  (deploying/waitress)
off-topic : stock market prices today         best 0.127  (tutorial/next)


## 6. Cost of exhaustive search

Every query touches all N vectors. Times below are the baseline that Task 5 compares FAISS and ChromaDB
against, and the reason ANN indexes exist at all.

In [14]:
Dt = torch.from_numpy(D).to(model.device)
qt = torch.from_numpy(q_demo).to(model.device)


def bench(fn, n=200):
    fn()                                                  # warm up
    if model.device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    if model.device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000          # ms per call


print(f"N = {len(chunks)} chunks, d = {D.shape[1]}")
print(f"  numpy  cosine + top-3 : {bench(lambda: top_k(cosine_similarity_numpy(q_demo, D), 3)):.3f} ms")
print(f"  torch  cosine (GPU)   : {bench(lambda: cosine_similarity_torch(qt, Dt)):.3f} ms")
print(f"  query embedding       : {bench(lambda: model.encode('How do I register a blueprint?'), n=20):.1f} ms")

N = 344 chunks, d = 384
  numpy  cosine + top-3 : 0.072 ms
  torch  cosine (GPU)   : 0.084 ms


  query embedding       : 5.3 ms


Two things in those numbers matter later:

- **Encoding the query costs ~70x more than searching all 344 chunks.** At this corpus size the vector
  search is not the bottleneck at all, so any "our search is fast" claim in Task 5 has to be read against
  a fixed ~5 ms embedding cost that no index can remove.
- **The GPU is slower than NumPy here.** 344x384 is far too small to cover kernel-launch and transfer
  overhead; the GPU only starts winning once N is large. Task 5 benchmarks N = 100 / 1000 / 5000, so this
  is the low end of a crossover we should be able to show explicitly.

## 7. The same code as a reusable module

The functions above also live in [`src/raghood/search.py`](../src/raghood/search.py), so Task 4 (chunk-size
sweeps, hybrid search, re-ranking) and Task 5 (framework benchmarks) import the verified implementation
instead of a retyped copy. Check that the module really does agree with what this notebook derived.

In [15]:
from raghood import search as S

assert np.allclose(S.cosine_similarity_numpy(q_demo, D), mine, atol=1e-7)
assert list(S.top_k(mine, 5)) == list(top_k(mine, 5))

hit = S.search("How do I register a blueprint?", model, D, chunks, k=3)[0]
print(f"raghood.search top hit: {hit['page']} > {hit['section']}  score {hit['score']:.4f}")
print("module matches the notebook's results")

raghood.search top hit: tutorial/views > Create a Blueprint  score 0.4864
module matches the notebook's results
